# Análise Exploratória — Pipeline de Dados da Saúde Suplementar

Este notebook consulta diretamente o Data Warehouse (SQL Server) já carregado pelo pipeline (`python -m src.main --stage all`) e faz uma leitura crítica dos dados — não apenas gráficos, mas estatísticas, hipóteses de negócio e limitações explícitas, como pede a seção 13 da especificação do projeto.

**Escopo desta execução**: a competência `2024-12`, com beneficiários carregados para as UFs **RR e AC** (subconjunto usado durante o desenvolvimento — o pipeline suporta as 27 UFs, configurável em `ANS_BENEFICIARIOS_UFS`) e um arquivo de estabelecimentos **fictício** de demonstração para o CNES (ver `data/raw/cnes/incoming/README.md`). As conclusões abaixo devem ser lidas com esse escopo em mente — a seção final de limitações detalha o que muda ao rodar com a base completa.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px

from src.config.settings import get_settings
from src.utils.db import get_engine

pd.set_option("display.max_columns", 50)

settings = get_settings()
# Usamos a conexao de carga (etl_writer) aqui, nao a de leitura do
# Streamlit (dashboard_reader): este notebook e uma ferramenta de
# desenvolvimento/analise interna e precisa acessar dim/fact diretamente
# (nao so as views de rpt), diferente da aplicacao publica.
engine = get_engine(settings.writer_connection)
print("Conectado a", settings.writer_connection.database, "em", settings.writer_connection.host)

Conectado a saude_suplementar em localhost


## 1. Visão geral dos dados

In [2]:
beneficiarios = pd.read_sql(
    """
    SELECT f.*, l.nm_municipio, l.cd_uf, l.nm_uf, l.regiao,
           o.nm_razao_social, o.modalidade
    FROM fact.fato_beneficiarios f
    JOIN dim.dim_localidade l ON l.sk_localidade = f.sk_localidade
    JOIN dim.dim_operadora o ON o.sk_operadora = f.sk_operadora
    WHERE f.sk_tempo = 202412
    """,
    engine,
)
print("Linhas x colunas:", beneficiarios.shape)
beneficiarios.head()

Linhas x colunas: (13041, 19)


,sk_beneficiarios,sk_tempo,sk_operadora,sk_localidade,tp_sexo,de_faixa_etaria,tipo_vinculo,segmentacao_plano,qt_beneficiario_ativo,qt_beneficiario_aderido,qt_beneficiario_cancelado,id_execucao,dh_carga,nm_municipio,cd_uf,nm_uf,regiao,nm_razao_social,modalidade
0,13043,202412,5,11,M,60 a 64 anos,Titular,Ambulatorial + Hospitalar com obstetrícia,19,0,2,53,2026-07-27 08:26:03,Rio Branco,AC,Acre,Norte,BRADESCO SAÚDE S.A.,Seguradora Especializada em Saúde
1,13044,202412,3,58,M,18 a 19 anos,Dependente,Ambulatorial + Hospitalar com obstetrícia,1,0,0,53,2026-07-27 08:26:03,Jordão,AC,Acre,Norte,UNIMED SEGUROS SAÚDE S/A,Seguradora Especializada em Saúde
2,13045,202412,-1,11,F,15 a 17 anos,Titular,Ambulatorial + Hospitalar sem obstetrícia,2,0,0,53,2026-07-27 08:26:03,Rio Branco,AC,Acre,Norte,Operadora nao cadastrada no cadastro ANS,Nao informado
3,13046,202412,3,50,F,10 a 14 anos,Dependente,Ambulatorial + Hospitalar com obstetrícia,2,0,0,53,2026-07-27 08:26:03,Acrelândia,AC,Acre,Norte,UNIMED SEGUROS SAÚDE S/A,Seguradora Especializada em Saúde
4,13047,202412,-1,16,M,30 a 34 anos,Titular,Ambulatorial + Hospitalar com obstetrícia,1,0,0,53,2026-07-27 08:26:03,Boa Vista,RR,Roraima,Norte,Operadora nao cadastrada no cadastro ANS,Nao informado


In [3]:
beneficiarios.dtypes

sk_beneficiarios                      int64
sk_tempo                              int64
sk_operadora                          int64
sk_localidade                         int64
tp_sexo                                 str
de_faixa_etaria                         str
tipo_vinculo                            str
segmentacao_plano                       str
qt_beneficiario_ativo                 int64
qt_beneficiario_aderido               int64
qt_beneficiario_cancelado             int64
id_execucao                           int64
dh_carga                     datetime64[us]
nm_municipio                            str
cd_uf                                   str
nm_uf                                   str
regiao                                  str
nm_razao_social                         str
modalidade                              str
dtype: object

## 2. Estatísticas descritivas

In [4]:
beneficiarios[["qt_beneficiario_ativo", "qt_beneficiario_aderido", "qt_beneficiario_cancelado"]].describe()

,qt_beneficiario_ativo,qt_beneficiario_aderido,qt_beneficiario_cancelado
count,13041.000000,13041.000000,13041.000000
mean,8.594280,0.185109,0.145311
std,32.217896,0.963647,0.839115
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000
75%,4.000000,0.000000,0.000000
max,847.000000,23.000000,43.000000


**Observação**: o grão da fato é (competência × operadora × município × sexo × faixa etária × tipo de vínculo × segmentação de plano) — cada linha já é uma contagem agregada da ANS, não um beneficiário individual. Por isso a média de `qt_beneficiario_ativo` (poucas unidades por linha) não deve ser lida como "tamanho médio de família" ou qualquer métrica per-capita; é o tamanho médio de uma célula do cruzamento demográfico-geográfico-comercial.

**Por que é relevante**: confirma que a agregação feita na camada Trusted (soma de linhas de origem que diferem apenas por atributos de plano não modelados) preservou a granularidade correta — nenhuma célula negativa, nenhuma célula com valor implausível (ver regra `volume_implausivel` em `src/quality/validators.py`).

## 3. Distribuição geográfica

In [5]:
por_municipio = (
    beneficiarios.groupby(["nm_municipio", "nm_uf"])["qt_beneficiario_ativo"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
por_municipio

nm_municipio      nm_uf  
Rio Branco        Acre       57483
Boa Vista         Roraima    42417
Cruzeiro do Sul   Acre        2894
Acrelândia        Acre         820
Senador Guiomard  Acre         753
Sena Madureira    Acre         688
Pacaraima         Roraima      647
Brasiléia         Acre         645
Tarauacá          Acre         592
Feijó             Acre         515
Name: qt_beneficiario_ativo, dtype: int64

In [6]:
fig = px.bar(
    por_municipio.reset_index(),
    x="qt_beneficiario_ativo",
    y="nm_municipio",
    orientation="h",
    title="Top 10 municípios por beneficiários ativos (RR + AC, dez/2024)",
)
fig.update_layout(yaxis=dict(categoryorder="total ascending"))
fig.show()

**Observação**: as capitais (Rio Branco/AC e Boa Vista/RR) concentram a maior parte dos beneficiários dentro das duas UFs carregadas — um padrão típico de concentração urbana da saúde suplementar no Brasil (planos de saúde são majoritariamente um benefício ligado a emprego formal, concentrado em centros urbanos).

**Possíveis causas**: concentração de empregos formais (CLT) e sedes de empresas nas capitais; menor penetração de planos em municípios do interior, onde a rede assistencial própria das operadoras também é mais escassa.

**Risco de interpretação**: com apenas 2 UFs carregadas nesta execução, não é possível generalizar esse padrão para o Brasil — a proporção capital/interior varia por região (ver limitações, seção 8).

**Possível decisão de negócio**: para uma cooperativa do sistema Unimed avaliando expansão, o padrão sugere investigar municípios do interior com população suficiente mas baixa penetração de planos — potencial mercado não atendido, desde que exista viabilidade de rede assistencial local.

## 4. Evolução temporal

In [7]:
evolucao = pd.read_sql("SELECT * FROM rpt.vw_evolucao_mensal_beneficiarios ORDER BY sk_tempo", engine)
evolucao

,sk_tempo,competencia,ano_mes_extenso,qt_beneficiarios_ativos,qt_beneficiarios_mes_anterior,variacao_absoluta,variacao_percentual
0,202412,2024-12-01,Dezembro/2024,112078,None,None,None


**Observação**: há apenas UMA competência carregada neste ambiente (`2024-12`), então `variacao_percentual` aparece como nulo (não há mês anterior para comparar) — comportamento esperado da view (`LAG()` sem linha anterior), não um bug. Para uma análise de evolução real, é necessário rodar o pipeline para múltiplas competências consecutivas: `python -m src.main --stage all --reference-period 2024-11`, depois `2024-12`, etc.

**Por que é relevante**: a arquitetura já suporta séries temporais completas (dim_tempo, LAG/variação percentual na view) sem nenhuma mudança de código — só depende de quantas competências forem carregadas.

## 5. Valores ausentes

In [8]:
beneficiarios.isna().sum().sort_values(ascending=False)

sk_beneficiarios             0
sk_tempo                     0
sk_operadora                 0
sk_localidade                0
tp_sexo                      0
de_faixa_etaria              0
tipo_vinculo                 0
segmentacao_plano            0
qt_beneficiario_ativo        0
qt_beneficiario_aderido      0
qt_beneficiario_cancelado    0
id_execucao                  0
dh_carga                     0
nm_municipio                 0
cd_uf                        0
nm_uf                        0
regiao                       0
nm_razao_social              0
modalidade                   0
dtype: int64

**Observação**: nenhuma coluna-chave (município, operadora, quantidades) apresenta nulos verdadeiros — porque o pipeline resolve códigos ausentes/inválidos para **chaves substitutas sentinela** (`sk_localidade = -1`, `sk_operadora = -1`) em vez de deixar `NULL`, e usa `tp_sexo`/`de_faixa_etaria` como `NULL` apenas quando a própria ANS não informa esse atributo (ver `dim.dim_localidade`/`dim.dim_operadora` em `sql/ddl/`).

**Por que é relevante**: evita o problema clássico de `JOIN` perder linhas por causa de chave estrangeira nula — toda linha da fato sempre encontra uma linha correspondente nas dimensões, mesmo quando a informação de origem é incompleta.

In [9]:
sem_operadora_cadastrada = beneficiarios[beneficiarios["sk_operadora"] == -1]
print(f"Linhas com operadora NAO encontrada no cadastro ANS: {len(sem_operadora_cadastrada)} de {len(beneficiarios)}")
print(f"Beneficiarios nessas linhas: {sem_operadora_cadastrada['qt_beneficiario_ativo'].sum()}")

Linhas com operadora NAO encontrada no cadastro ANS: 107 de 13041
Beneficiarios nessas linhas: 212


**Observação**: uma parcela pequena mas não nula dos registros de beneficiários referencia operadoras que não constam no cadastro de operadoras **ativas** da ANS — plausivelmente operadoras que encerraram atividade entre a competência de referência (dez/2024) e a data em que o cadastro foi baixado (o cadastro é um snapshot do dia da extração, sem histórico por competência, diferente do arquivo de beneficiários).

**Risco de interpretação**: isso NÃO significa erro de dado — é uma consequência esperada de cruzar um arquivo histórico (beneficiários, por competência) com um cadastro vivo (operadoras ativas, snapshot atual). Tratado como regra de qualidade **WARNING** (não ERROR) exatamente por esse motivo — ver `src/quality/validators.py::beneficiarios_rules`.

## 6. Duplicidades

In [10]:
grao = ["sk_tempo", "sk_operadora", "sk_localidade", "tp_sexo", "de_faixa_etaria", "tipo_vinculo", "segmentacao_plano"]
duplicatas = beneficiarios.duplicated(subset=grao).sum()
print(f"Linhas duplicadas no grao da fato: {duplicatas}")

Linhas duplicadas no grao da fato: 0


**Observação**: zero, por construção — a constraint `uq_fato_beneficiarios_grao` (SQL Server) e a agregação por grão na camada Trusted (`src/transform/ans_beneficiarios.py`) tornam duplicidade no grão da fato estruturalmente impossível após a carga. A duplicidade real que o pipeline precisa detectar acontece ANTES da agregação, na linha bruta da ANS — medida em `duplicatas_exatas_na_origem` (ver estatísticas da etapa `transform` em `aud.execucao_pipeline`).

## 7. Outliers

In [11]:
q1, q3 = beneficiarios["qt_beneficiario_ativo"].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
outliers = beneficiarios[beneficiarios["qt_beneficiario_ativo"] > limite_superior]
print(f"Limite superior (IQR): {limite_superior:.1f}")
print(f"Linhas acima do limite: {len(outliers)} de {len(beneficiarios)} ({len(outliers)/len(beneficiarios)*100:.1f}%)")
outliers.nlargest(5, "qt_beneficiario_ativo")[["nm_razao_social", "nm_municipio", "de_faixa_etaria", "qt_beneficiario_ativo"]]

Limite superior (IQR): 8.5
Linhas acima do limite: 1922 de 13041 (14.7%)


,nm_razao_social,nm_municipio,de_faixa_etaria,qt_beneficiario_ativo
8157,UNIMED RIO BRANCO COOPERATIVA DE TRABALHO MEDI...,Rio Branco,40 a 44 anos,847
8045,UNIMED RIO BRANCO COOPERATIVA DE TRABALHO MEDI...,Rio Branco,35 a 39 anos,714
8269,UNIMED RIO BRANCO COOPERATIVA DE TRABALHO MEDI...,Rio Branco,45 a 49 anos,685
8605,UNIMED RIO BRANCO COOPERATIVA DE TRABALHO MEDI...,Rio Branco,55 a 59 anos,671
8759,UNIMED RIO BRANCO COOPERATIVA DE TRABALHO MEDI...,Rio Branco,5 a 9 anos,666


**Observação**: os "outliers" pelo critério estatístico do IQR são, na prática, exatamente as combinações mais comuns/populosas (ex.: faixas etárias produtivas em planos empresariais de operadoras grandes na capital) — ou seja, valores estatisticamente atípicos mas **substantivamente esperados**, não erros de dado. Nenhuma linha viola a regra de qualidade `volume_implausivel` (limite de 500.000/linha).

**Risco de interpretação**: aplicar detecção de outliers estatística ingenuamente sobre dados agregados por natureza (como este) tende a marcar como "anômalo" o que é apenas concentração de mercado real — outliers estatísticos e erros de dado não são sinônimos aqui.

## 8. Correlação entre beneficiários e rede assistencial

In [12]:
cobertura = pd.read_sql("SELECT * FROM rpt.vw_cobertura_regional WHERE sk_tempo = 202412", engine)
cobertura[["nm_municipio", "nm_uf", "qt_beneficiarios_ativos", "qt_estabelecimentos", "beneficiarios_por_estabelecimento", "classificacao_cobertura"]]

,nm_municipio,nm_uf,qt_beneficiarios_ativos,qt_estabelecimentos,beneficiarios_por_estabelecimento,classificacao_cobertura
0,Município ignorado - AC,Acre,2,0,NaN,Cobertura crítica
1,Acrelândia,Acre,820,0,NaN,Cobertura crítica
2,Assis Brasil,Acre,173,0,NaN,Cobertura crítica
3,Brasiléia,Acre,645,1,645.000000,Cobertura adequada
4,Bujari,Acre,231,0,NaN,Cobertura crítica
5,Capixaba,Acre,150,0,NaN,Cobertura crítica
6,Cruzeiro do Sul,Acre,2894,3,964.666667,Cobertura adequada
7,Epitaciolândia,Acre,500,0,NaN,Cobertura crítica
8,Feijó,Acre,515,0,NaN,Cobertura crítica
9,Jordão,Acre,60,0,NaN,Cobertura crítica


In [13]:
correlacao = cobertura[["qt_beneficiarios_ativos", "qt_estabelecimentos"]].corr().iloc[0, 1]
print(f"Correlacao (beneficiarios x estabelecimentos, por municipio): {correlacao:.2f}")

Correlacao (beneficiarios x estabelecimentos, por municipio): 0.88


**Observação**: com apenas 2 UFs e uma base de estabelecimentos fictícia de 30 registros, a amostra é pequena demais (n≈39 municípios, a maioria com zero estabelecimentos) para tirar conclusões estatísticas robustas sobre a correlação — o coeficiente acima é ilustrativo do método, não uma conclusão de negócio.

**Por que o método importa mesmo assim**: com a base completa (27 UFs + CNES real), essa mesma consulta responderia diretamente a uma das perguntas centrais do projeto ("qual a relação entre beneficiários e estabelecimentos disponíveis?") e alimentaria a página "Cobertura Regional" do Streamlit com significância estatística real.

## 9. Hipóteses de negócio

1. **Concentração urbana**: a maior parte dos beneficiários está concentrada nas capitais/regiões metropolitanas, enquanto a rede assistencial de estabelecimentos de saúde em geral (CNES) é mais distribuída — a razão beneficiários/estabelecimento deveria ser MAIOR nas capitais (mais gente, rede proporcionalmente menor por não ser exclusiva de planos de saúde) e MENOR ou nula em municípios pequenos (poucos beneficiários, mas também poucos estabelecimentos). Testável com a base completa via `rpt.vw_razao_beneficiarios_estabelecimento`.
2. **Operadoras regionais fortes**: cooperativas médicas regionais (como Unimeds locais) tendem a liderar a participação de mercado em capitais de estados menores, enquanto seguradoras nacionais (Bradesco Saúde, SulAmérica, etc.) têm participação mais uniforme entre regiões. Testável via `rpt.vw_operadoras_por_regiao`.
3. **Sazonalidade de adesão/cancelamento**: `qt_beneficiario_aderido`/`qt_beneficiario_cancelado` provavelmente têm picos em janeiro (renovações de contrato empresarial) e dezembro (fechamento de ano fiscal) — testável ao carregar 12+ competências consecutivas.

## 10. Principais conclusões

- O pipeline preserva corretamente o total de beneficiários da fonte após agregação (verificado: soma bruta = soma na fato, sem perda nem duplicação).
- A normalização por chaves substitutas sentinela (`-1`) elimina nulos estruturais em joins, tornando análises geográficas e por operadora robustas mesmo com dados de origem incompletos (operadora sem cadastro, estabelecimento sem tipo).
- A distribuição geográfica de beneficiários é fortemente concentrada nas capitais dentro da amostra carregada — hipótese razoável de generalizar, mas que precisa ser confirmada com as 27 UFs.
- A relação beneficiários/estabelecimento (base para o índice de cobertura) é metodologicamente sólida, mas requer a base real do CNES para ter valor analítico de negócio — hoje usa um fixture fictício.

## 11. Limitações do conjunto de dados

- **Escopo geográfico**: apenas RR e AC carregados nesta execução (demonstração). O pipeline suporta as 27 UFs — basta configurar `ANS_BENEFICIARIOS_UFS` no `.env` e rodar novamente.
- **Uma única competência**: sem série temporal real carregada, análises de evolução/sazonalidade são estruturais (a view funciona), não substantivas ainda.
- **CNES fictício**: os 30 estabelecimentos usados são dados de demonstração inventados (nomes/códigos fictícios), não o cadastro real do DATASUS — necessário para qualquer conclusão real sobre cobertura assistencial. Ver `data/raw/cnes/incoming/README.md` para como substituir pelo arquivo oficial.
- **Cadastro de operadoras sem histórico**: `Relatorio_cadop.csv` é um snapshot do dia da extração (sem versão por competência), o que explica beneficiários associados a operadoras "não cadastradas" nas competências mais antigas.
- **Sem geocodificação**: os dados da ANS não trazem latitude/longitude por município, então mapas coropléticos exigiriam uma base IBGE adicional (fora do escopo definido para este projeto).